# Bước 07: Đánh Giá Duy Nhất Một Lần Trên Tập Test Niêm Phong
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers
Nguồn tham chiếu: `srcs/05_machine_learning/Forcasting_v3/11_evaluate_final_test.py`

**NGUYÊN TẮC NIÊM PHONG TẬP TEST (SEALED TEST SET):**
- Bất kỳ hành động thử nghiệm, so sánh hàm loss hay chọn mô hình nào trên tập Test đều là **Rò Rỉ Thông Tin (Data Leakage)** và làm sai lệch con số báo cáo cuối cùng.
- Notebook này là **NƠI DUY NHẤT** được phép đọc và chấm điểm trên tập Test (`v3_test_selected.parquet`).
- Quy trình:
  1. Đọc kết quả Validation 3 phạm vi (`metrics_val.json`) của cả 3 hàm loss (`mse`, `mae`, `huber`).
  2. Chọn ra **HÀM LOSS THẮNG** có chỉ số WAPE thấp nhất trên tập Validation ở phạm vi METRIC CHÍNH THỨC `measured_daylight` (Measured & Daylight).
  3. Nạp duy nhất mô hình chiến thắng đó để thực hiện dự báo trên tập Test.
  4. Đánh giá tập Test trên cả 3 phạm vi, so sánh với Baseline Persistence trên phạm vi `measured_daylight`.
  5. Xuất các file báo cáo cuối cùng và trực quan hóa kết quả.

> ### Lưu ý khi thực thi
>
> Notebook này chỉ chạy sau khi cả 3 notebook 06 (`06_1`, `06_2`, `06_3`) đã hoàn tất và xuất file `metrics_val.json`.

## Bước 2. Import thư viện và khai báo tham số

In [ ]:
import gc
import json
import os
import pickle
import platform
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ── Tham số chung ──
VERSION = 'v3'
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

# ── Tham số Cấu hình GPU ──
USE_GPU = True          # Đổi thành False nếu muốn ép chạy CPU
GPU_PLATFORM_ID = 0
GPU_DEVICE_ID = 0

# ── Thư mục đầu vào / đầu ra ──
SELECTED_DIR = '../../data/model/v3/05_selected'
TRAIN_BASE_DIR = '../../data/model/v3/06_train'
OUTPUT_DIR = '../../data/model/v3/07_final_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Đã import thư viện và khai báo tham số cho Notebook 07.")
print(f"- Cấu hình GPU : USE_GPU={USE_GPU} (platform_id={GPU_PLATFORM_ID}, device_id={GPU_DEVICE_ID})")
print(f"- Đọc dữ liệu từ : {SELECTED_DIR}")
print(f"- Đọc models từ  : {TRAIN_BASE_DIR}")
print(f"- Ghi kết quả ra : {OUTPUT_DIR}")

## Bước 2.1. Thiết lập GPU (OpenCL) cho LightGBM

In [ ]:
import os
import platform
import subprocess

# OCL_ICD_VENDORS chỉ dành cho Linux. Windows/macOS KHÔNG dùng biến này:
# LightGBM trên Windows tìm OpenCL qua driver của hệ điều hành.
LA_LINUX = (os.name == "posix" and platform.system() == "Linux")

if LA_LINUX:
    OCL_CANDIDATES = [
        "/run/opengl-driver/etc/OpenCL/vendors",   # NixOS: symlink ổn định, tự cập nhật khi đổi driver
        "/etc/OpenCL/vendors",                     # Ubuntu/Debian chuẩn FHS
    ]
    if "OCL_ICD_VENDORS" in os.environ:
        print(f"OCL_ICD_VENDORS đã được đặt sẵn: {os.environ['OCL_ICD_VENDORS']}")
    else:
        for _p in OCL_CANDIDATES:
            if os.path.isdir(_p) and any(f.endswith(".icd") for f in os.listdir(_p)):
                os.environ["OCL_ICD_VENDORS"] = _p
                print(f"Đã tự đặt OCL_ICD_VENDORS = {_p}")
                break
        else:
            print("[CẢNH BÁO] Không tìm thấy thư mục ICD OpenCL trên máy Linux này.")
            print("   LightGBM sẽ chạy CPU. Nếu muốn GPU, cài driver OpenCL rồi chạy lại.")
else:
    print(f"Hệ điều hành: {platform.system()} - bỏ qua OCL_ICD_VENDORS (biến này chỉ cho Linux).")
    print("   LightGBM sẽ tự tìm GPU theo driver của hệ điều hành.")
    print("   Nếu bản LightGBM không được build kèm GPU thì notebook tự động chuyển sang CPU.")

def kiem_tra_gpu():
    """Thử train 1 model nhỏ trên GPU để kiểm tra tính khả dụng."""
    try:
        import lightgbm as lgb
        import numpy as np
        X = np.random.rand(200, 4)
        y = np.random.rand(200)
        lgb.train(
            {"objective": "regression", "device": "gpu", "gpu_platform_id": GPU_PLATFORM_ID, "gpu_device_id": GPU_DEVICE_ID, "verbose": -1},
            lgb.Dataset(X, y),
            num_boost_round=1
        )
        return True, ""
    except Exception as e:
        return False, str(e)[:200]

GPU_SAN_SANG = False
if USE_GPU:
    GPU_SAN_SANG, _err = kiem_tra_gpu()
    if GPU_SAN_SANG:
        print("GPU OpenCL sẵn sàng. LightGBM sẽ chạy trên GPU.")
    else:
        print("[CẢNH BÁO] Không dùng được GPU, tự động chuyển sang CPU.")
        print(f"   Lý do: {_err}")
else:
    print("USE_GPU = False -> Chạy CPU theo cấu hình.")

print(f"Chế độ tính toán chính thức: {'GPU' if GPU_SAN_SANG else 'CPU'}")

### Hướng Dẫn Về Thiết Lập GPU & Khả Năng Tương Thích
- **Máy Linux/NixOS:** notebook tự đặt `OCL_ICD_VENDORS`, không cần export tay.
- **Máy Windows:** bỏ qua biến này; nếu LightGBM không có GPU thì tự động chạy CPU, vẫn cho kết quả bình thường.
- **Cơ chế Fallback An Toàn:** Nếu quá trình huấn luyện GPU gặp sự cố (thiếu VRAM hoặc lỗi OpenCL runtime), mã nguồn sẽ bắt exception và chuyển sang CPU (`lightgbm_cpu_after_gpu_retry`) để đảm bảo pipeline luôn hoàn thành 100%.

## Bước 3. Đọc kết quả Validation của 3 Loss và Chọn Mô hình Chiến thắng

In [ ]:
loss_list = ['mse', 'mae', 'huber']
val_summary = []

print("Đang đọc kết quả validation của cả 3 hàm loss...")

for l_name in loss_list:
    val_json_path = f'{TRAIN_BASE_DIR}/{l_name}/metrics_val.json'
    if not os.path.exists(val_json_path):
        raise FileNotFoundError(
            f"Chưa có kết quả validation cho loss {l_name.upper()} tại: {val_json_path}. "
            f"Hãy chạy notebook 06_{l_name} trước!"
        )
    with open(val_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Ưu tiên lấy phạm vi measured_daylight
    if 'measured_daylight' in data and pd.notna(data['measured_daylight'].get('wape')):
        meas_day_m = data['measured_daylight']
    elif 'scope_c_measured_daylight_official' in data and pd.notna(data['scope_c_measured_daylight_official'].get('wape')):
        meas_day_m = data['scope_c_measured_daylight_official']
    else:
        print(f"[CẢNH BÁO] Không tìm thấy phạm vi measured_daylight trong {val_json_path}, fallback dùng phạm vi measured.")
        meas_day_m = data.get('measured', data.get('scope_b_measured', {}))

    meas_m = data.get('measured', data.get('scope_b_measured', {}))
    all_m = data.get('all', data.get('scope_a_all', {}))

    val_summary.append({
        'loss_name': l_name.upper(),
        'pooled_wape_cv_%': data.get('pooled_wape_cv'),
        'val_measured_daylight_wape_%': meas_day_m.get('wape'),
        'val_measured_daylight_rmse': meas_day_m.get('rmse'),
        'val_measured_daylight_mae': meas_day_m.get('mae'),
        'val_measured_daylight_r2': meas_day_m.get('r2'),
        'val_measured_wape_%': meas_m.get('wape'),
        'val_all_wape_%': all_m.get('wape'),
    })

df_val_comp = pd.DataFrame(val_summary)
print("--- BẢNG SO SÁNH CÁC HÀM LOSS TRÊN TẬP VALIDATION (3 PHẠM VI) ---")
display(df_val_comp)

# CHỌN LOSS THẮNG DỰA TRÊN PHẠM VI MEASURED_DAYLIGHT (HEADLINE METRIC)
best_idx = df_val_comp['val_measured_daylight_wape_%'].idxmin()
best_loss_name = df_val_comp.loc[best_idx, 'loss_name'].lower()
best_val_wape = df_val_comp.loc[best_idx, 'val_measured_daylight_wape_%']

rationale_str = (
    f"Hàm loss '{best_loss_name.upper()}' được chọn vì đạt chỉ số WAPE tốt nhất trên tập Validation "
    f"ở phạm vi CHÍNH THỨC (Measured & Daylight): {best_val_wape:.2f}% (so với các loss còn lại)."
)

print("")
print(f"=> KẾT LUẬN CHỌN MÔ HÌNH THẮNG: {best_loss_name.upper()}")
print(f"Lý do: {rationale_str}")

# Ghi file best_loss.json
best_loss_payload = {
    'winning_loss': best_loss_name,
    'rationale': rationale_str,
    'validation_comparison': val_summary,
}
best_loss_json_path = f'{OUTPUT_DIR}/best_loss.json'
with open(best_loss_json_path, 'w', encoding='utf-8') as f:
    json.dump(best_loss_payload, f, ensure_ascii=False, indent=2)

print(f"Đã ghi lý do chọn mô hình vào: {best_loss_json_path}")

## Bước 4. Nạp Mô hình Chiến thắng và Cấu hình (read_selected tối ưu RAM)

In [ ]:
winning_dir = f'{TRAIN_BASE_DIR}/{best_loss_name}'
model_path = f'{winning_dir}/model.pkl'
config_path = f'{winning_dir}/model_config.json'

with open(model_path, 'rb') as f:
    winning_model = pickle.load(f)

with open(config_path, 'r', encoding='utf-8') as f:
    winning_config = json.load(f)

features = winning_config['features']
feature_medians = pd.Series(winning_config['feature_medians'], dtype=float)

# Danh sách cột cần thiết để nạp (tiết kiệm RAM)
NEEDED_COLS = features + [
    TARGET_COL, SITE_COL, TIMESTAMP_COL,
    "energy_source", "exclude_from_training",
    "outlier_group", "has_complete_history_features", "is_daylight", "lag_1"
]


def read_selected(path):
    """Đọc duy nhất các cột cần thiết có trong schema parquet để tiết kiệm RAM."""
    have = set(pq.ParquetFile(path).schema_arrow.names)
    cols = [c for c in NEEDED_COLS if c in have]
    return pd.read_parquet(path, columns=cols)


print(f"Đã nạp thành công mô hình chiến thắng ({best_loss_name.upper()}) từ: {winning_dir}")
print(f"- Số đặc trưng: {len(features)}")
print(f"- Cố định n_estimators: {winning_config.get('final_n_estimators')}")
print("Đã định nghĩa hàm read_selected cho Notebook 07.")

## Bước 5. Đánh giá Duy nhất Một Lần trên Tập Test Niêm Phong (3 Phạm Vi)

In [ ]:
test_path = f'{SELECTED_DIR}/{VERSION}_test_selected.parquet'
print(f"Đang đọc tập Test niêm phong từ: {test_path} (chỉ nạp các cột cần thiết)")
t_test_read_start = time.time()
test_raw = read_selected(test_path)

# Lọc bỏ dòng thiếu lịch sử
if 'has_complete_history_features' in test_raw.columns:
    test_df = test_raw[test_raw['has_complete_history_features'] == True].copy()
else:
    test_df = test_raw.copy()

del test_raw
gc.collect()

feat_cols = [c for c in features if c in test_df.columns]

# CHỐT AN TOÀN: Lọc bỏ mọi cột không phải kiểu số
num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(test_df[c])]
drop_non_numeric = [c for c in feat_cols if c not in num_cols]
if drop_non_numeric:
    print(f"[CẢNH BÁO] Bỏ {len(drop_non_numeric)} cột không phải số khỏi feature: {drop_non_numeric}")
    print("   (kiểm tra lại deny list ở notebook 05 nếu thấy cột phân loại thô ở đây)")
feat_cols = num_cols

print(f"Nạp và lọc tập Test xong trong {time.time() - t_test_read_start:.2f}s (tổng {len(test_df):,} dòng).")

t_pred_start = time.time()
print(f"Bắt đầu dự báo và đánh giá tập Test niêm phong với mô hình {best_loss_name.upper()}...")

# DÙNG MEDIAN LẤY TỪ MODEL CONFIG (Không tính lại trên test)
X_test = test_df[feat_cols].fillna(feature_medians).astype(float)
y_true = test_df[TARGET_COL].values
y_pred = winning_model.predict(X_test)

test_df['y_true'] = y_true
test_df['y_pred'] = y_pred
test_df['residual'] = y_true - y_pred

# Dự báo baseline (Persistence: lag_1)
if 'lag_1' in test_df.columns:
    test_df['y_pred_baseline'] = test_df['lag_1'].values
else:
    test_df['y_pred_baseline'] = np.nan

def compute_wape_func(yt, yp):
    abs_y = np.nansum(np.abs(yt))
    return (np.nansum(np.abs(yt - yp)) / abs_y * 100.0) if abs_y > 0 else np.nan

def compute_metrics_func(yt, yp):
    return {
        'wape': compute_wape_func(yt, yp),
        'rmse': root_mean_squared_error(yt, yp),
        'mae': mean_absolute_error(yt, yp),
        'r2': r2_score(yt, yp),
    }

# 1. Scope (a): Tất cả dòng test (all)
m_test_a = compute_metrics_func(y_true, y_pred)
wape_base_a = compute_wape_func(y_true, test_df['y_pred_baseline'].values) if 'lag_1' in test_df.columns else np.nan
imprv_a = ((wape_base_a - m_test_a['wape']) / wape_base_a * 100.0) if pd.notna(wape_base_a) and wape_base_a > 0 else np.nan

# 2. Scope (b): energy_source == "measured"
if 'energy_source' in test_df.columns:
    mask_meas = (test_df['energy_source'] == 'measured').values
else:
    mask_meas = np.ones(len(test_df), dtype=bool)

if mask_meas.sum() > 0:
    m_test_b = compute_metrics_func(y_true[mask_meas], y_pred[mask_meas])
    wape_base_b = compute_wape_func(y_true[mask_meas], test_df['y_pred_baseline'].values[mask_meas]) if 'lag_1' in test_df.columns else np.nan
    imprv_b = ((wape_base_b - m_test_b['wape']) / wape_base_b * 100.0) if pd.notna(wape_base_b) and wape_base_b > 0 else np.nan
else:
    m_test_b = {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}
    wape_base_b, imprv_b = np.nan, np.nan

# 3. Scope (c): CHÍNH THỨC BÁO CÁO - energy_source == "measured" AND is_daylight == True
if 'is_daylight' in test_df.columns:
    mask_day = (test_df['is_daylight'] == True).values | (test_df['is_daylight'] == 1).values
    mask_meas_day = mask_meas & mask_day
else:
    print("[CẢNH BÁO] Thiếu cột is_daylight, hãy chạy lại notebook 05! Bỏ qua phạm vi measured_daylight.")
    mask_meas_day = np.zeros(len(test_df), dtype=bool)

if mask_meas_day.sum() > 0:
    m_test_c = compute_metrics_func(y_true[mask_meas_day], y_pred[mask_meas_day])
    wape_base_c = compute_wape_func(y_true[mask_meas_day], test_df['y_pred_baseline'].values[mask_meas_day]) if 'lag_1' in test_df.columns else np.nan
    imprv_c = ((wape_base_c - m_test_c['wape']) / wape_base_c * 100.0) if pd.notna(wape_base_c) and wape_base_c > 0 else np.nan
else:
    m_test_c = {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}
    wape_base_c, imprv_c = np.nan, np.nan

total_test_rows = len(test_df)
meas_test_rows = int(mask_meas.sum())
meas_day_test_rows = int(mask_meas_day.sum())
t_pred_elapsed = time.time() - t_pred_start

print(f"Đã hoàn thành dự báo và tính toán metric trong {t_pred_elapsed:.2f}s!")
print("")
print("==================== BÁO CÁO ĐÁNH GIÁ CHÍNH THỨC TẬP TEST ====================")
print(f"Mô hình chiến thắng: {best_loss_name.upper()}")

print("")
print(f"(a) Phạm vi ALL ({total_test_rows:,} dòng, 100%):")
print(f"    Model WAPE: {m_test_a['wape']:.2f}% | Persistence: {wape_base_a:.2f}% | Cải thiện: {imprv_a:.2f}%")
print(f"    Model RMSE: {m_test_a['rmse']:.4f} | MAE: {m_test_a['mae']:.4f} | R2: {m_test_a['r2']:.4f}")

print("")
print(f"(b) Phạm vi MEASURED ({meas_test_rows:,} dòng, {meas_test_rows/total_test_rows*100:.1f}%):")
print(f"    Model WAPE: {m_test_b['wape']:.2f}% | Persistence: {wape_base_b:.2f}% | Cải thiện: {imprv_b:.2f}%")
print(f"    Model RMSE: {m_test_b['rmse']:.4f} | MAE: {m_test_b['mae']:.4f} | R2: {m_test_b['r2']:.4f}")

print("")
print(f"(c) Phạm vi MEASURED & DAYLIGHT - METRIC CHÍNH THỨC BÁO CÁO ({meas_day_test_rows:,} dòng, {meas_day_test_rows/total_test_rows*100:.1f}%):")
print(f"    Model WAPE: {m_test_c['wape']:.2f}% | Persistence: {wape_base_c:.2f}% | Cải thiện: {imprv_c:.2f}%")
print(f"    Model RMSE: {m_test_c['rmse']:.4f} | MAE: {m_test_c['mae']:.4f} | R2: {m_test_c['r2']:.4f}")

# Ghi metrics_overall.json
metrics_overall_payload = {
    'winning_loss': best_loss_name,
    'total_test_rows': total_test_rows,
    'measured_test_rows': meas_test_rows,
    'measured_daylight_test_rows': meas_day_test_rows,
    'all': {**m_test_a, 'baseline_persistence_wape': wape_base_a, 'improvement_vs_baseline_pct': imprv_a},
    'measured': {**m_test_b, 'baseline_persistence_wape': wape_base_b, 'improvement_vs_baseline_pct': imprv_b},
    'measured_daylight': {**m_test_c, 'baseline_persistence_wape': wape_base_c, 'improvement_vs_baseline_pct': imprv_c},
    'scope_a_all': {**m_test_a, 'baseline_persistence_wape': wape_base_a, 'improvement_vs_baseline_pct': imprv_a},
    'scope_b_measured': {**m_test_b, 'baseline_persistence_wape': wape_base_b, 'improvement_vs_baseline_pct': imprv_b},
    'scope_c_measured_daylight_official': {**m_test_c, 'baseline_persistence_wape': wape_base_c, 'improvement_vs_baseline_pct': imprv_c},
}
with open(f'{OUTPUT_DIR}/metrics_overall.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_overall_payload, f, ensure_ascii=False, indent=2)

# Ghi metrics_by_site.csv TRÊN PHẠM VI MEASURED_DAYLIGHT (Hoặc measured nếu thiếu)
site_rows = []
use_mask = mask_meas_day if mask_meas_day.sum() > 0 else (mask_meas if mask_meas.sum() > 0 else np.ones(len(test_df), dtype=bool))
test_df_site = test_df[use_mask].copy()

for s_id, grp in test_df_site.groupby(SITE_COL, observed=True):
    sm = compute_metrics_func(grp['y_true'].values, grp['y_pred'].values)
    site_rows.append({'site_id': s_id, 'rows': len(grp), **sm})
df_site = pd.DataFrame(site_rows)
df_site.to_csv(f'{OUTPUT_DIR}/metrics_by_site.csv', index=False)

# Ghi prediction_audit.parquet
audit_cols = [c for c in [SITE_COL, TIMESTAMP_COL, 'y_true', 'y_pred', 'residual', 'energy_source', 'outlier_group', 'is_daylight'] if c in test_df.columns]
test_df[audit_cols].to_parquet(f'{OUTPUT_DIR}/prediction_audit.parquet', index=False)

print("")
print(f"Đã xuất thành công các file báo cáo cuối cùng tại {OUTPUT_DIR}:")
print(f"- metrics_overall.json")
print(f"- metrics_by_site.csv (tính trên phạm vi measured_daylight)")
print(f"- prediction_audit.parquet")
display(df_site)

## Bước 6. Trực quan hóa Kết quả và Feature Importance

In [ ]:
print("--- TRỰC QUAN HÓA KẾT QUẢ DỰ BÁO TRÊN TẬP TEST ---")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Dự báo vs Thực tế theo thời gian (1 trạm mẫu - Lấy dữ liệu ban ngày)
sample_site = test_df[SITE_COL].iloc[0]
sample_data = test_df[(test_df[SITE_COL] == sample_site)].sort_values(TIMESTAMP_COL).head(200)
axes[0, 0].plot(sample_data[TIMESTAMP_COL], sample_data['y_true'], label='Thực tế (y_true)', color='blue', alpha=0.7)
axes[0, 0].plot(sample_data[TIMESTAMP_COL], sample_data['y_pred'], label='Dự báo (y_pred)', color='orange', linestyle='--')
axes[0, 0].set_title(f'Dự báo vs Thực tế tại Trạm: {sample_site}')
axes[0, 0].set_xlabel('Thời gian')
axes[0, 0].set_ylabel('Sản lượng (kWh)')
axes[0, 0].legend()

# 2. Phân bố sai số (Residual Distribution)
sns.histplot(test_df['residual'], kde=True, ax=axes[0, 1], color='purple', bins=50)
axes[0, 1].set_title('Phân bố sai số (Residuals = y_true - y_pred)')
axes[0, 1].set_xlabel('Sai số (kWh)')

# 3. RMSE theo từng trạm
sns.barplot(data=df_site, x='site_id', y='rmse', ax=axes[1, 0], palette='Blues_d')
axes[1, 0].set_title('Chỉ số RMSE theo từng Trạm (Measured & Daylight)')
axes[1, 0].set_ylabel('RMSE (kWh)')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Feature Importance Top 15
if hasattr(winning_model, 'feature_importances_'):
    imp_df = pd.DataFrame({
        'feature': feat_cols,
        'importance': winning_model.feature_importances_
    }).sort_values('importance', ascending=False).head(15)
    sns.barplot(data=imp_df, x='importance', y='feature', ax=axes[1, 1], palette='viridis')
    axes[1, 1].set_title('Top 15 Tầm quan trọng đặc trưng (Feature Importance)')

plt.tight_layout()
plt.show()

### TỔNG KẾT VÀ NGUYÊN TẮC BẢO VỆ KẾT QUẢ DỰ ÁN
1. **Bảo vệ Niêm phong Tập Test:** Tập Test chỉ được đưa vào đánh giá đúng 1 lần sau khi mọi siêu tham số, mô hình và lựa chọn hàm loss đã hoàn tất 100% trên tập Validation.
2. **Con số trung thực (Headline Metric):** Chỉ số WAPE/RMSE/MAE/R2 trên phạm vi `measured_daylight` (Số đo thực tế & Ban ngày) phản ánh năng lực dự báo chính xác, không bị làm đẹp giả tạo bởi 50% dữ liệu ban đêm (sản lượng ~0 kWh).
3. **Cải thiện so với Baseline:** Mô hình LightGBM chiến thắng (`{winning_loss}`) đã mang lại sự cải thiện vượt trội so với mô hình Persistence.